# Backpropagation Lab

This notebook trains and tunes `MLPClassifier`/`MLPRegressor` (multi-layer perceptron) models from scikit-learn, covering:

- Baseline training and diagnosing overfitting (Iris dataset)
- Overfitting mitigation: early stopping with a validation set, and L2 loss regularization
- Systematic hyperparameter sweeps: learning rate, hidden layer width, and momentum (Vowel dataset)
- Automated hyperparameter search via `GridSearchCV` and `RandomizedSearchCV`, compared for speed and accuracy
- Applying `MLPRegressor` to a real-valued time-series prediction task on a large (700K+ row) dataset

Datasets are loaded directly from public URLs so the notebook runs end-to-end without local setup, aside from the large regression dataset noted below.

In [ ]:
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.model_selection import train_test_split

In [ ]:
import urllib.request, io

def load_arff_url(url):
    """Load an .arff file directly from a URL (no local download needed)."""
    text = urllib.request.urlopen(url).read().decode('utf-8')
    return arff.loadarff(io.StringIO(text))


## Avoiding Overfit: Early Stopping and Loss Regularization

### Baseline: No Overfit Avoidance
I trained an `MLPClassifier` on the Iris dataset with one-hot encoded outputs (3 output nodes, one per class), using:
- `hidden_layer_sizes = [64]` — one hidden layer, 64 nodes
- `activation = 'logistic'`
- `solver = 'sgd'`
- `alpha = 0`
- `batch_size = 1`
- `learning_rate_init = 0.01`
- `shuffle = True`
- `momentum = 0`
- `n_iter_no_change = 50`
- `max_iter = 10000`

I used a random 80/20 train/test split, ran it across a few different random splits, and averaged the number of iterations to converge along with training and test accuracy. For one run I also inspected the softmax probabilities on the test set via `predict_proba`.

In [ ]:
#Iris with no regularization
data, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/iris.arff')

iris = pd.DataFrame(data)
iris['class'] = iris['class'].str.decode('utf-8')

X = iris[['petallength', 'petalwidth', 'sepallength', 'sepalwidth']]
y = pd.get_dummies(iris['class'])

mlp = MLPClassifier(hidden_layer_sizes = [64], activation = 'logistic', solver = 'sgd', alpha = 0, batch_size = 1, learning_rate_init = 0.01, shuffle = True,
                    momentum = 0, n_iter_no_change = 50, max_iter = 10000)

def mlp_func(random):
  X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=random)
  mlp.fit(X_train, y_train)
  n_iter = mlp.n_iter_
  train_score = mlp.score(X_train, y_train)
  test_score = mlp.score(X_test, y_test)
  return n_iter, train_score, test_score

n_iters = []
train_scores = []
test_scores = []

for i in range(0,4):
  n_iter, train_score, test_score = mlp_func(i)
  n_iters.append(n_iter)
  train_scores.append(train_score)
  test_scores.append(test_score)

mean_n = np.mean(n_iters)
mean_train = np.mean(train_scores)
mean_test = np.mean(test_scores)

print(f'The average number of iterations until convergence is {mean_n}')
print(f'The average training score accuracy is {mean_train}')
print(f'The average testing score accuracy is {mean_test}')

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=0)
mlp.fit(X_train, y_train)
probabilities = mlp.predict_proba(X_test)
print("\nSoftmax probabilities for the final run's test set:")
print(probabilities)

**Analysis**

It took about 1 minute to run through 4 iterations of different random splits of the model. My average accuracy for the training set was 0.973 and my average accuracy for the testing set was 0.967. I actually do not see any evidence of overfitting as the train and test scores are nearly identical, though this dataset is very simplistic with a lack of much noise.

### Early Stopping with a Validation Set
I repeated the same setup, this time adding early stopping via the `early_stopping`, `validation_fraction`, and `n_iter_no_change` parameters, holding out a validation set from the training data. Across several random splits I averaged the number of iterations to converge, training accuracy, test accuracy, and the model's best validation score, and for one run plotted validation accuracy against epoch.

Note: since Iris is a small, low-noise dataset, early stopping doesn't show the accuracy gains it typically would on noisier data — holding out a validation set from an already-small training set adds variance rather than helping.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes = [64], activation = 'logistic', solver = 'sgd', alpha = 0, batch_size = 1, learning_rate_init = 0.01, shuffle = True,
                    momentum = 0, n_iter_no_change = 50, max_iter = 10000, early_stopping = True, validation_fraction = 0.1)

def mlp_func(random):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=random)
    mlp.fit(X_train, y_train)
    n_iter = mlp.n_iter_
    train_score = mlp.score(X_train, y_train)
    test_score = mlp.score(X_test, y_test)
    best_val_score = mlp.best_validation_score_
    return n_iter, train_score, test_score, best_val_score

n_iters = []
train_scores = []
test_scores = []
best_val_scores = []

for i in range(0,4):
    n_iter, train_score, test_score, best_val_score = mlp_func(i)
    n_iters.append(n_iter)
    train_scores.append(train_score)
    test_scores.append(test_score)
    best_val_scores.append(best_val_score)

mean_n = np.mean(n_iters)
mean_train = np.mean(train_scores)
mean_test = np.mean(test_scores)
mean_val = np.mean(best_val_scores)

print(f'The average number of iterations until convergence is {mean_n}')
print(f'The average training score accuracy is {mean_train}')
print(f'The average testing score accuracy is {mean_test}')
print(f'The average best validation score is {mean_val}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=0)
mlp.fit(X_train, y_train)

plt.plot(mlp.validation_scores_)
plt.xlabel('Epochs')
plt.ylabel('Validation Set Accuracy')
plt.title('Validation Set Accuracy vs Epochs')
plt.show()

**Analysis**

Now my average number of iterations until convergence is around 111. Using the validation set and early stopping actually decreased my training and testing set accuracies down to 0.902 and 0.892 respectively, but I think this can be attributed to the fact that the iris dataset's simplicity and lack of noise. This was mentioned in the 1.2 note and it makes complete sense. The validation scores graph increased with more epochs and then became highly volatile, swinging between roughly 0.33 and 1.0 before the model finally stopped training.

### L2 Loss Regularization
Returning to the baseline setup (no early stopping), I instead varied the L2 regularization strength (`alpha`), trying values from 0.00001 up to 0.1, and recorded the number of iterations to converge, training/test accuracy, and best loss value for each. For the best-performing alpha, I plotted the training loss curve against epoch.

In [ ]:
alphas = [0.1, 0.01, 0.001, 0.0001, 0.00001]

results = []

for alpha in alphas:
    mlp = MLPClassifier(hidden_layer_sizes = [64], activation = 'logistic', solver = 'sgd', alpha = alpha, batch_size = 1, learning_rate_init = 0.01, shuffle = True,
                        momentum = 0, n_iter_no_change = 50, max_iter = 10000)
    n_iters = []
    train_scores = []
    test_scores = []
    best_losses = []
    for i in range(0,4):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=i)
        mlp.fit(X_train, y_train)
        n_iters.append(mlp.n_iter_)
        train_scores.append(mlp.score(X_train, y_train))
        test_scores.append(mlp.score(X_test, y_test))
        best_losses.append(mlp.best_loss_)
    results.append([alpha, np.mean(n_iters), np.mean(train_scores), np.mean(test_scores), np.mean(best_losses)])

results_df = pd.DataFrame(results, columns = ['Alpha', 'Avg Iterations', 'Avg Train Accuracy', 'Avg Test Accuracy', 'Avg Best Loss'])
print(results_df)

print('The best alpha value for me was the smallest, which was 0.0001')

mlp_best = MLPClassifier(hidden_layer_sizes=[64], activation='logistic', solver='sgd',
                         alpha=0.0001, batch_size=1, learning_rate_init=0.01,
                         shuffle=True, momentum=0, n_iter_no_change=50, max_iter=10000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
mlp_best.fit(X_train, y_train)

plt.plot(mlp_best.loss_curve_)
plt.title(f'Training Loss vs Epochs (Alpha = 0.0001)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

**Analysis**

L2 regularization (specifically with an alpha of 0.0001) gave me the best results, achieving a test accuracy of 98.3% and slightly outperforming the baseline model with no regularization (96.7%). Because the Iris dataset is simple and lacks noise, the baseline model generalized well on its own, but a small L2 penalty provided just enough restriction to optimize the weights further. Early stopping, however, was the least effective method and dropped the test accuracy down to around 89%. This decrease happened because holding out a 10% validation set from an already small dataset did not give the model enough training data, leading to early stopping based on a high-variance validation score.

## Hyperparameter Tuning on the Vowel Dataset

### Baseline Considerations
Before tuning, I worked out the baseline (majority-class) accuracy for both Iris and the [Vowel dataset](https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/vowel.arff), considered why Vowel is a harder classification problem than Iris, and identified which input features shouldn't be used for training.

1) The baseline accuracy for iris is roughly 33% as there are 3 classes and vowel is roughly 9% since there are 11 classes.
2) The vowel data set will probably have lower accuracy because it is much harder to classify an instance when there are 11 classes to choose from vs only 3. It is also very easy for neural networks to make distinctions between the highly distinct measurements from the iris dataset.
3) The Speaker, Sex, and Train/Test input features should be dropped so the model doesn't cheat by memorizing the people instead of the sound waves.

### Learning Rate
I loaded the Vowel dataset, dropped the Speaker, Sex, and Train/Test columns identified above, and decoded/encoded the remaining categorical features. Using one hidden layer sized at twice the number of input features, a 75/25 train/test split, and no early stopping, I swept 5+ learning rates (from 0.001 up to 10) and recorded epochs-to-convergence, training accuracy, and test accuracy for each in a table.

In [ ]:
# Train with different learning rates
data, meta = load_arff_url('https://raw.githubusercontent.com/cs472ta/CS472/master/datasets/vowel.arff')

vowel = pd.DataFrame(data)
vowel['Sex'] = vowel['Sex'].str.decode('utf-8')
vowel = vowel.replace('Male', 0)
vowel = vowel.replace('Female', 1)

vowel['Class'] = vowel['Class'].str.decode('utf-8')

X = vowel.drop(columns=['Class', 'Train or Test', 'Speaker Number', 'Sex'])
y = pd.get_dummies(vowel['Class'])

n_inputs = X.shape[1]
n_hidden = n_inputs * 2

learning_rates = [0.001, 0.01, 0.1, 1, 10]

results = []

for lr in learning_rates:
    mlp = MLPClassifier(hidden_layer_sizes = [n_hidden], activation = 'logistic', solver = 'sgd', alpha = 0, batch_size = 1, learning_rate_init = lr, shuffle = True,
                        momentum = 0, n_iter_no_change = 50, max_iter = 1000)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=0)
    mlp.fit(X_train, y_train)

    n_iter = mlp.n_iter_
    train_score = mlp.score(X_train, y_train)
    test_score = mlp.score(X_test, y_test)

    results.append([lr, n_iter, train_score, test_score])

results_df = pd.DataFrame(results, columns = ['Learning Rate', 'Epochs', 'Train Accuracy', 'Test Accuracy'])
print(results_df)

**Analysis**

A learning rate of 0.01 yielded the highest test accuracy at roughly 80.6%. Learning rates that were too high (1.0 and 10.0) resulted in much faster training times (fewer epochs) but terrible accuracy. the smallest learning rate (0.001) also resulted in low accuracy because it learned too slowly and hit the maximum iteration cap of 1000 epochs before it could fully converge. The 0.01 rate provided the best balance of steady learning within the allotted training time.

### Number of Hidden Nodes
Using the best learning rate from above, I swept the number of hidden nodes — starting at 1, then doubling (2, 4, 8, …) — until additional nodes stopped improving accuracy, recording the same metrics in a table.

In [ ]:
hidden_nodes = [1, 2, 4, 8, 16, 32, 64, 128]

results = []

for n_hidden in hidden_nodes:
    mlp = MLPClassifier(hidden_layer_sizes = [n_hidden], activation = 'logistic', solver = 'sgd', alpha = 0, batch_size = 1, learning_rate_init = 0.01, shuffle = True,
                        momentum = 0, n_iter_no_change = 50, max_iter = 1000)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=0)
    mlp.fit(X_train, y_train)
    n_iter = mlp.n_iter_
    train_score = mlp.score(X_train, y_train)
    test_score = mlp.score(X_test, y_test)
    results.append([n_hidden, n_iter, train_score, test_score])

results_df = pd.DataFrame(results, columns = ['Hidden Nodes', 'Epochs', 'Train Accuracy', 'Test Accuracy'])
print(results_df)

**Analysis**

Increasing the number of hidden nodes steadily improved the model's accuracy but **significantly** increased the training time. With only 1 to 4 nodes, the network couldnt learn the data very well, resulting in poor accuracy and early stopping. As the hidden nodes increased to 64 and 128, the model gained enough capacity to perfectly fit the training data (1.0 accuracy) and achieve a high test accuracy of ~90%. these larger networks required more epochs to converge rhough, hitting the 1000 max_iter limit. The larger weight matrices and reaching the max epochs  is what caused it take so long (took 20 minutes to fully run).

### Momentum
With the best learning rate and hidden-node count fixed, I swept 5+ momentum values between 0 and 1, again recording epochs-to-convergence and train/test accuracy for each.

In [ ]:
# Train with different momentum values
momentum_values = [0, 0.2, 0.4, 0.6, 0.8, 0.9]

results = []

for momentum in momentum_values:
    mlp = MLPClassifier(hidden_layer_sizes = [n_hidden], activation = 'logistic', solver = 'sgd', alpha = 0, batch_size = 1, learning_rate_init = 0.01, shuffle = True,
                        momentum = momentum, n_iter_no_change = 50, max_iter = 1000)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=0)
    mlp.fit(X_train, y_train)
    n_iter = mlp.n_iter_
    train_score = mlp.score(X_train, y_train)
    test_score = mlp.score(X_test, y_test)
    results.append([momentum, n_iter, train_score, test_score])

results_df = pd.DataFrame(results, columns = ['Momentum', 'Epochs', 'Train Accuracy', 'Test Accuracy'])
print(results_df)

**Analysis**

Adding momentum affected both training time and accuracy. For training time, higher momentum values (0.8 and 0.9) allowed the model to converge faster, which dropped the required epochs from the 1000 limit down to 733 and 578. For accuracy, a smaller momentum value of 0.2 yielded the best result at roughly 93.5%. higher values like 0.8 caused a decrease in performance.

### Automated Hyperparameter Search
Rather than tuning learning rate, hidden nodes, and momentum manually, I used `GridSearchCV` and `RandomizedSearchCV` to search over the most promising values from my manual experiments above, comparing runtime and the resulting best hyperparameters/accuracy for each method.

In [ ]:
#Grid search for hyperparameters.
#Here is one variation of code you could use for your grid search. You can try your own variation if you prefer.
from sklearn.model_selection import GridSearchCV

clf = MLPClassifier(activation='logistic', solver='sgd', alpha=0, early_stopping=True, n_iter_no_change=10)
parameters = {'learning_rate_init': [0.001, 0.01, 0.1],
    'hidden_layer_sizes': [[32], [64], [128]],
    'momentum': [0.0, 0.2, 0.4]}

grid = GridSearchCV(clf, parameters)
grid.fit(X, y)

print(grid.best_params_)
print(grid.best_score_)

In [ ]:
#Randomized search for hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
clf = MLPClassifier(activation='logistic', solver='sgd', alpha=0, early_stopping=True, n_iter_no_change=10)
distributions = dict(
    learning_rate_init=uniform(loc=0.001, scale=0.099),
    hidden_layer_sizes=[[32], [64], [128]],
    momentum=uniform(loc=0.0, scale=0.99)
)
search = RandomizedSearchCV(clf, distributions, n_iter=10)
search.fit(X, y)

print(search.best_params_)
print(search.best_score_)

**Analysis**

The grid search took 23 minutes, the randomized parameter took 9 minutes to run. Randomized is much more efficient but grid search is technically more accurate. Grid Search should be used for small parameter lists or when you have plenty of computing time. Use Randomized Search when you have a massive range of options or want to avoid waiting a long time.

## Regression with MLPs

### Applying MLPRegressor to a Real-World Dataset
For a regression task, I used `MLPRegressor` (linear output activation, SSE loss) instead of `MLPClassifier` (softmax activation, cross-entropy loss) on a real-world dataset with a continuous target (see below). I normalized the input features, split the data into train/test sets, and reported MAE for both.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

# Dataset: hourly Amazon/Google product review counts (time series).
# Place 'y_amazon-google-large.csv' in the same directory as this notebook to rerun.
data = pd.read_csv('y_amazon-google-large.csv')

df = pd.DataFrame(data).dropna().sample(n=100000, random_state=42)
df['ds'] = pd.to_datetime(df['ds'])
df['year'] = df['ds'].dt.year
df['month'] = df['ds'].dt.month
df['day'] = df['ds'].dt.day
df['hour'] = df['ds'].dt.hour

X = df.drop(columns=['ds', 'y', 'unique_id'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
mlp = MLPRegressor(hidden_layer_sizes = [64], activation = 'relu', solver = 'adam', alpha = 0.0001, learning_rate_init = 0.01, shuffle = True,
                   momentum = 0, n_iter_no_change = 50, max_iter = 1000)

mlp.fit(X_train, y_train)

train_mae = mean_absolute_error(mlp.predict(X_train), y_train)
test_mae = mean_absolute_error(mlp.predict(X_test), y_test)

print(f'The training set MAE is {train_mae}')
print(f'The testing set MAE is {test_mae}')

**Analysis**

I used the Amazon/Google Reviews dataset to predict hourly review countsand got a training and testing MAE of roughly 180. This was when I sampled data however, when I let it run on the whole thing it got a training and testing MAE of 145, but it took over an hour to run. While the MLPClassifier uses softmax and cross-entropy loss to categorize data, the MLPRegressor uses a linear output and MAE loss to predict continuous numbers. I used the adam solver with default batch sizes to prevent exploding gradients and massive training times, although even then it still took quite some computing time.

### Additional Hyperparameters
On the same dataset, I compared different hidden-layer activation functions (logistic, tanh, relu) and tested adding multiple hidden layers, then briefly explored solver choice (sgd vs. adam), batch size, and learning-rate adaptation schedules.

In [ ]:
# Run with different hyperparameters

activations = ['logistic', 'tanh', 'relu']
results = []
for activation in activations:
    mlp = MLPRegressor(hidden_layer_sizes = [64], activation = activation, solver = 'adam', alpha = 0.0001, learning_rate_init = 0.01, shuffle = True,
                       momentum = 0, n_iter_no_change = 50, max_iter = 1000)
    mlp.fit(X_train, y_train)
    train_mae = mean_absolute_error(mlp.predict(X_train), y_train)
    test_mae = mean_absolute_error(mlp.predict(X_test), y_test)
    results.append([activation, train_mae, test_mae])
results_df = pd.DataFrame(results, columns = ['Activation', 'Train MAE', 'Test MAE'])
print(results_df)

hidden_layers = [[64], [64, 64], [64, 64, 64]]
results = []
for layers in hidden_layers:
    mlp = MLPRegressor(hidden_layer_sizes = layers, activation = 'relu', solver = 'adam', alpha = 0.0001, learning_rate_init = 0.01, shuffle = True,
                       momentum = 0, n_iter_no_change = 50, max_iter = 1000)
    mlp.fit(X_train, y_train)
    train_mae = mean_absolute_error(mlp.predict(X_train), y_train)
    test_mae = mean_absolute_error(mlp.predict(X_test), y_test)
    results.append([str(layers), train_mae, test_mae])

results_df = pd.DataFrame(results, columns = ['Hidden Layers', 'Train MAE', 'Test MAE'])
print(results_df)

**Analysis**

the tanh and relu activations slightly outperformed logistic, while adding multiple hidden layers generally increases the network's learning capacity up to a point of diminishing returns. The choice of solver and batch size drastically impacts efficiency, as using sgd with a batch size of 1 previously caused massive training delays and exploding errors. Switching to an adaptive solver like adam and utilizing standard batch sizes smoothed out the weight updates which also allowed the model to converge safely and quicker.